In [2]:
# scripts/01c_ckan_dataset_discovery.py
import requests
import json

# ==============================================================================
# MILESTONE 2.1c: CKAN DATASET DISCOVERY
# Goal: Find all constraint-related datasets and their resources
# ==============================================================================

API_URL = "https://api.neso.energy/api/3/action"

def search_all_constraint_datasets():
    """
    Search for all datasets related to constraints, balancing, and boundaries.
    """
    search_terms = [
        "constraint",
        "balancing",
        "boundary",
        "curtailment",
        "BOALF",
        "BOA",
        "acceptance",
        "transmission",
        "SCOTEX",
        "SSEN"
    ]
    
    all_packages = {}
    
    print("="*80)
    print("CKAN DATASET DISCOVERY")
    print("="*80)
    
    for term in search_terms:
        print(f"\nSearching for: '{term}'")
        
        params = {
            "q": term,
            "rows": 50
        }
        
        response = requests.get(f"{API_URL}/package_search", params=params)
        response.raise_for_status()
        
        data = response.json()
        
        if data.get("success"):
            results = data["result"]["results"]
            print(f"  Found {len(results)} datasets")
            
            for pkg in results:
                pkg_id = pkg["id"]
                if pkg_id not in all_packages:
                    all_packages[pkg_id] = {
                        "title": pkg["title"],
                        "name": pkg.get("name", ""),
                        "resources": []
                    }
                    
                    for res in pkg.get("resources", []):
                        all_packages[pkg_id]["resources"].append({
                            "id": res["id"],
                            "name": res.get("name", ""),
                            "format": res.get("format", ""),
                            "description": res.get("description", "")[:100]
                        })
    
    # Print summary
    print("\n" + "="*80)
    print("DISCOVERED DATASETS SUMMARY")
    print("="*80)
    
    for i, (pkg_id, info) in enumerate(all_packages.items(), 1):
        print(f"\n{i}. {info['title']}")
        print(f"   Package ID: {pkg_id}")
        print(f"   Resources ({len(info['resources'])}):")
        for res in info['resources']:
            print(f"     - {res['name']}")
            print(f"       ID: {res['id']}")
            print(f"       Format: {res['format']}")
            if res['description']:
                print(f"       Desc: {res['description']}")
    
    # Save to file for reference
    with open("data/intermediate/ckan_discovered_datasets.json", "w") as f:
        json.dump(all_packages, f, indent=2)
    
    print(f"\n✅ Full results saved to: data/intermediate/ckan_discovered_datasets.json")
    
    return all_packages

def check_specific_known_datasets():
    """
    Check some known NESO dataset IDs that might contain constraint data.
    """
    print("\n" + "="*80)
    print("CHECKING SPECIFIC KNOWN DATASETS")
    print("="*80)
    
    # These are common NESO dataset patterns - we'll check if they exist
    known_resource_ids = [
        # Constraint-related
        "24d067d8-1328-452a-9720-21cb691e491e",  # We know this one exists (Constraint Breakdown)
        # Add more if you find them in the NESO data portal
    ]
    
    for resource_id in known_resource_ids:
        print(f"\nChecking resource: {resource_id}")
        try:
            params = {
                "resource_id": resource_id,
                "limit": 1
            }
            response = requests.get(f"{API_URL}/datastore_search", params=params)
            response.raise_for_status()
            
            data = response.json()
            if data.get("success"):
                records = data["result"]["records"]
                if records:
                    print(f"  ✅ EXISTS - Columns: {list(records[0].keys())}")
                else:
                    print(f"  ⚠️ EXISTS but empty")
            else:
                print(f"  ❌ Does not exist or access denied")
        except Exception as e:
            print(f"  ❌ Error: {e}")

if __name__ == "__main__":
    import os
    os.makedirs("data/intermediate", exist_ok=True)
    
    # Run comprehensive search
    discovered = search_all_constraint_datasets()
    
    # Check specific known datasets
    check_specific_known_datasets()
    
    print("\n" + "="*80)
    print("NEXT STEPS:")
    print("1. Review the discovered datasets above")
    print("2. Look for datasets with 'BOALF', 'Balancing Mechanism', or 'Constraint' in the name")
    print("3. Identify which resource IDs contain boundary-level data")
    print("4. Update the reconciliation spike with the correct resource ID")
    print("="*80)

CKAN DATASET DISCOVERY

Searching for: 'constraint'
  Found 14 datasets

Searching for: 'balancing'
  Found 42 datasets

Searching for: 'boundary'
  Found 10 datasets

Searching for: 'curtailment'
  Found 1 datasets

Searching for: 'BOALF'
  Found 0 datasets

Searching for: 'BOA'
  Found 1 datasets

Searching for: 'acceptance'
  Found 16 datasets

Searching for: 'transmission'
  Found 25 datasets

Searching for: 'SCOTEX'
  Found 3 datasets

Searching for: 'SSEN'
  Found 0 datasets

DISCOVERED DATASETS SUMMARY

1. Thermal Constraint Costs
   Package ID: f0055054-c55c-4068-a01c-61da4334e58f
   Resources (11):
     - Thermal Constraint Costs 19-20
       ID: d195f1d8-7d9e-46f1-96a6-4251e75e9bd0
       Format: XLSX
       Desc: Out turn system costs for thermal constraints across a number of significant constraint boundaries f
     - Thermal Constraint Costs 20-21
       ID: a4302916-85af-4a4d-8171-1bdf8f0697a7
       Format: XLSX
       Desc: Out turn system costs for thermal constraints 